In [ ]:
# ── 第一格：环境准备（D02 封装之后，每本 notebook 开头都是这十几行）──
%load_ext autoreload
%autoreload 2
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from signal_insight import config, geo, quality, viz
from signal_insight.viz import MAIN, MAIN_LIGHT, MAIN_DARK, MAIN_MID, GRAY, GRAY_DARK, RED, AMBER, INK, MUTED, FAINT   # 配色常量

print("绘图字体：", " → ".join(viz.apply_theme()))  # 自动识别已安装字体及常规/粗体
con = sqlite3.connect(config.DB_PATH)      # 路径从包里来：notebook 在哪个文件夹打开都找得到库
DAY = config.ANALYSIS_DAY                  # 分析日期（口径常量）
NB = "D06_驻留识别"
SOURCE = f"{config.SOURCE_NOTE}；脚本：{NB}"   # 每张图左下角的来源行
pd.set_option("display.max_columns", 20)

In [ ]:
hot = pd.read_sql("SELECT * FROM dim_hotspot", con).iloc[0]
DAY_START = pd.Timestamp(DAY)
def mask_id(imsi):
    """展示用户时：只显示IMSI尾号，不在图和表里暴露完整ID"""
    return "尾号" + str(imsi)[-4:]

In [ ]:
dwd = pd.read_sql("SELECT imsi, ts, lac_cell FROM dwd_signal",
                con, parse_dates=['ts'])
dim = pd.read_sql("SELECT lac_cell, site_id, lng, lat FROM dim_cell", con)
sig = (dwd.merge(dim, on='lac_cell', how='inner')).sort_values(
    ['imsi','ts'],ignore_index=True
)
sig['到西塔(米)'] = geo.haversine(sig['lat'], sig['lng'], hot['lat'], hot['lng'])
n_users = sig['imsi'].nunique()

In [ ]:
h = sig["ts"].dt.hour
night_far = sig[(h == 3) & (sig["到西塔(米)"] > 3000)]["imsi"].unique()
evening_in = sig[(h.between(19, 20)) &  # 19–20 点在西塔 500 米内
            (sig["到西塔(米)"] <= 500)]["imsi"].unique()
n_rec = sig.groupby("imsi").size()
candidates = n_rec[n_rec.index.isin(night_far) &
                n_rec.index.isin(evening_in) & n_rec.between(60, 120)]
demo_id = candidates.index[0]   # 按 IMSI 排序取第一个，保证每次选到同一个人
demo = sig[sig["imsi"] == demo_id]
print(f"候选 {len(candidates)} 人；示例用户 {mask_id(demo_id)}，当天 {len(demo)} 条信令")

In [ ]:
hour = (demo["ts"] - DAY_START).dt.total_seconds() / 3600          # 时间换算成“几点”（小数）
km = demo["到西塔(米)"] / 1000
arrive = demo[(demo["ts"].dt.hour >= 12) &  # 到达西塔的时间
            (demo["到西塔(米)"] <= 500)]["ts"].min()
night_km = km[demo["ts"].dt.hour < 6].median()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.6))
ax.axhspan(0, 0.5, color=MAIN_LIGHT, lw=0, zorder=0)                     # 西塔 500 米范围
ax.text(0.2, 0.25, "西塔 500 米内", va="center", color=MAIN, fontsize=9.5)
ax.plot(hour, km, color=GRAY, lw=1, zorder=1)
ax.scatter(hour, km, s=16, color=MAIN, zorder=2)
ax.annotate(f"{arrive:%H:%M} 到达西塔", (hour[demo["ts"] == arrive].iloc[0], 0.5), xytext=(25, 70),
            textcoords="offset points", ha="left", fontsize=10, color=INK, arrowprops=dict(arrowstyle="->", color=FAINT))
ax.set_xticks(range(0, 25, 3), [f"{x} 点" for x in range(0, 25, 3)])
ax.set_xlim(0, 24)
ax.set_ylim(0, km.max() * 1.15)
ax.set_ylabel("离西塔的距离（公里）", color=MUTED)

title = f"{mask_id(demo_id)} 的一天：凌晨在距西塔约 {night_km:.1f} 公里处，{arrive:%H:%M} 到达西塔"
viz.finish(fig, title,
           f"2018-10-03 该用户每条信令所连站点到西塔中心的距离；共 {len(demo)} 条；口径：dwd_signal × dim_cell",
           SOURCE, "D6_01_一个人的一天.png")
plt.show()